In [1]:
import json
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from tqdm import tqdm

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
OUTPUT_PATH = "./results/basesft.jsonl"

# load dataset
public_data = [json.loads(line) for line in open("./data/public.jsonl")]

n_mcq  = sum(bool(d.get("options")) for d in public_data)
n_free = sum(not d.get("options")   for d in public_data)
print(f"Loaded {len(public_data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

Loaded 1126 questions  (375 MCQ, 751 free-form)


In [2]:
# prompts for free response and MCQ problems
SYSTEM_PROMPT_FRQ = (
    "You are an expert mathematician. "
    "Solve the problem carefully and put your final answer within \boxed{}."
    "If there are multiple answers, put them in a single \\boxed{} separated by commas."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Solve the problem and choose the single best answer. "
    "At the end, output exactly one line in this form: \\boxed{A}"
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """ determine if free response or MCQ problem andeturn (system_prompt, user_prompt)"""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_FRQ, question

In [3]:
# # build SFT training data from the public dataset
# import json
# def normalize_answer_field(answer):
#     if isinstance(answer, list):
#         if len(answer) == 1:
#             return str(answer[0]).strip()
#         return ", ".join(str(x).strip() for x in answer)
#     return str(answer).strip()

# def build_user_prompt(question, options):
#     labels = [chr(65 + i) for i in range(len(options))]
#     opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
#     return f"{question}\n\nOptions:\n{opts_text}"

# input_path = "./data/public.jsonl"
# output_path = "./data/sft_train.jsonl"

# with open(input_path, "r") as fin, open(output_path, "w") as fout:
#     for line in fin:
#         item = json.loads(line)

#         question = item["question"].strip()
#         options = item.get("options")

#         is_mcq = options is not None and len(options) > 0

#         if is_mcq:
#             system_prompt = SYSTEM_PROMPT_MCQ
#             user_prompt = build_user_prompt(question, options)

#             # assumes answer is something like "B" or ["B"]
#             gold = normalize_answer_field(item["answer"]).upper()

#             assistant_text = f"\\boxed{{{gold}}}"
#         else:
#             system_prompt = SYSTEM_PROMPT_FRQ
#             user_prompt = question

#             gold = normalize_answer_field(item["answer"])
#             assistant_text = f"\\boxed{{{gold}}}"

#         record = {
#             "messages": [
#                 {"role": "system", "content": system_prompt},
#                 {"role": "user", "content": user_prompt},
#                 {"role": "assistant", "content": assistant_text},
#             ]
#         }

#         fout.write(json.dumps(record, ensure_ascii=False) + "\n")

# print(f"Wrote SFT data to {output_path}")

In [4]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig

dataset = load_dataset("json", data_files="./data/sft_train.jsonl", split="train")
dataset = dataset.select(range(min(100, len(dataset))))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)

# Important for QLoRA / k-bit training
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

# Attach adapters explicitly
model = get_peft_model(model, peft_config)

# Optional sanity check
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir="./qwen_math_sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=200,
    bf16=True,
    report_to="none",
    dataset_text_field="text",   # change if your dataset is not a text column
    max_length=1024,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model("./qwen_math_sft/test")
tokenizer.save_pretrained("./qwen_math_sft/test")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


Step,Training Loss
10,1.197200


('./qwen_math_sft/test/tokenizer_config.json',
 './qwen_math_sft/test/special_tokens_map.json',
 './qwen_math_sft/test/chat_template.jinja',
 './qwen_math_sft/test/vocab.json',
 './qwen_math_sft/test/merges.txt',
 './qwen_math_sft/test/added_tokens.json',
 './qwen_math_sft/test/tokenizer.json')

In [10]:
# # try this maybe
# import torch
# from datasets import load_dataset
# from transformers import (
#     AutoModelForCausalLM,
#     AutoTokenizer,
#     BitsAndBytesConfig,
# )
# from peft import (
#     LoraConfig,
#     get_peft_model,
#     prepare_model_for_kbit_training,
# )
# from trl import SFTTrainer, SFTConfig

# MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"

# # Load dataset
# dataset = load_dataset("json", data_files="./data/sft_train.jsonl", split="train")
# dataset = dataset.select(range(min(100, len(dataset))))

# # Tokenizer
# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# # 4-bit quantization config for QLoRA
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True,
# )

# # Load quantized base model
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     trust_remote_code=True,
#     quantization_config=bnb_config,
#     device_map="auto",
# )

# # Prepare for k-bit training
# model = prepare_model_for_kbit_training(model)

# # LoRA adapters
# peft_config = LoraConfig(
#     r=16,
#     lora_alpha=32,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules="all-linear",
# )

# # Explicitly attach adapters
# model = get_peft_model(model, peft_config)
# model.print_trainable_parameters()

# # SFT config
# training_args = SFTConfig(
#     output_dir="./qwen_math_sft",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,
#     learning_rate=2e-4,
#     num_train_epochs=2,
#     logging_steps=10,
#     save_steps=200,
#     bf16=True,
#     report_to="none",

#     # Your dataset is conversational "messages"
#     max_length=1024,

#     # Important for chat SFT
#     assistant_only_loss=True,
# )

# trainer = SFTTrainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset,
#     processing_class=tokenizer,
# )

# trainer.train()
# trainer.save_model("./qwen_math_sft/test")
# tokenizer.save_pretrained("./qwen_math_sft/test")

In [5]:
# load model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=True,
    enable_lora=True,   
    gpu_memory_utilization=0.88,
    max_model_len=16384, # 16384
    trust_remote_code=True,
    max_num_seqs=4, # 256
    max_num_batched_tokens=16384, # was 32768
)

sampling_params = SamplingParams(
    max_tokens=8192, # was 32768
    temperature=0,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")

INFO 04-14 19:36:45 [utils.py:233] non-default args: {'trust_remote_code': True, 'load_format': 'bitsandbytes', 'max_model_len': 16384, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.88, 'max_num_batched_tokens': 16384, 'max_num_seqs': 4, 'disable_log_stats': True, 'quantization': 'bitsandbytes', 'enable_lora': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}
INFO 04-14 19:36:46 [model.py:549] Resolved architecture: Qwen3ForCausalLM
INFO 04-14 19:36:46 [model.py:1678] Using max model len 16384
INFO 04-14 19:36:46 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-14 19:36:47 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-14 19:36:48 [system_utils.py:152] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized; WSL is detected and NVML is not c

(EngineCore pid=7566) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=7566) <frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:00,  2.05it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:01<00:00,  1.96it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.88it/s]
(EngineCore pid=7566) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 8/8 [00:03<00:00,  2.65it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 6/6 [00:00<00:00,  8.66it/s]


WARNING 04-14 19:37:30 [interface.py:525] Using 'pin_memory=False' as WSL is detected. This may slow down the performance.
Model loaded.


In [6]:
# Build prompts for first 10 entries
prompts = []
for item in public_data[:10]:
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    prompts.append(prompt_text)

# Generate
print(f"Generating responses for {len(prompts)} questions...")
outputs = llm.generate(prompts, sampling_params=sampling_params, lora_request=LoRARequest("math_sft", 1, "./qwen_math_sft/test"))

responses = [out.outputs[0].text.strip() for out in outputs]

Generating responses for 10 questions...


Rendering prompts:   0%|          | 0/10 [00:00<?, ?it/s]

WARNING 04-14 19:37:30 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [7]:
def extract_letter(text: str) -> str:
    matches = re.findall(r"\\boxed\{([A-Za-z])\}", text)
    if matches:
        return matches[-1].upper()

    m = re.search(r"answer\s+is\s+([A-Za-z])", text, re.IGNORECASE)
    if m:
        return m.group(1).upper()

    m = re.search(r"^\s*([A-Z])\s*$", text.strip(), re.MULTILINE)
    if m:
        return m.group(1).upper()

    return ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


# Load Judger for free-form scoring
sys.path.insert(0, ".")
from judger import Judger
judger = Judger(strict_extract=False)

results = []
for item, response in tqdm(zip(public_data, responses), total=len(public_data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]

    if is_mcq:
        correct = score_mcq(response, str(gold))
    else:
        gold_list = gold if isinstance(gold, list) else [gold]
        try:
            correct = judger.auto_judge(
                pred=response,
                gold=gold_list,
                options=[[]] * len(gold_list),
            )
        except Exception:
            correct = False

    results.append({
        "id":       item.get("id"),
        "is_mcq":   is_mcq,
        "gold":     gold,
        "response": response,
        "correct":  correct,
    })

print(f"Scoring complete. {len(results)} results.")

Scoring:   1%|          | 10/1126 [00:02<05:25,  3.43it/s]

Scoring complete. 10 results.


In [8]:
mcq_res  = [r for r in results if r["is_mcq"]]
free_res = [r for r in results if not r["is_mcq"]]

def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("EVALUATION RESULTS")
print("=" * 50)
print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({acc(results):.2f}%)")
print("=" * 50)

EVALUATION RESULTS
  MCQ        :    1 /    3  (33.33%)
  Free-form  :    2 /    7  (28.57%)
  Overall    :    3 /   10  (30.00%)


In [9]:
SAVE_EVAL = True   # Set to False when running on the private test set

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for r in results:
        if SAVE_EVAL:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "gold": r["gold"],
                      "response": r["response"], "correct": r["correct"]}
        else:
            record = {"id": r["id"], "is_mcq": r["is_mcq"], "response": r["response"]}
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(results)} records to {out_path}")

Saved 10 records to results/basesft.jsonl
